### Building A Simple RAG

__Note the LLM used here is deepseek__

In [30]:
#import the necessary dependedncies
import json
import requests 
from minsearch import AppendableIndex
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [ ]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [3]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [4]:
question = 'Can I still join the course?'

In [5]:
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [6]:
search_results = search(question)

In [7]:
prompt = build_prompt(question, search_results)

In [101]:
client = OpenAI(base_url="https://api.deepseek.com")

def llm(prompt):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "You are a helpful assistant"},
            {"role": "user", "content": prompt},
        ]
    )
    return response.choices[0].message.content

In [23]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [24]:
rag("When will the course start")

'The course will start on 15th Jan 2024 at 17h00 with the first "Office Hours" live session. \n\nYou can also subscribe to the course\'s public Google Calendar (desktop only), register before the start date, join the Telegram channel for announcements, and register in DataTalks.Club\'s Slack to stay updated.'

In [25]:
rag("How do I patch KDE under FreeBSD?")

'Based on the provided context, there is no information available regarding how to patch KDE under FreeBSD. Please consult the official FreeBSD documentation, forums, or other relevant resources for guidance on this topic.'

### Building a Simple Agentic RAG

In [26]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
{question}
</QUESTION>

<CONTEXT> 
{context}
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}
""".strip()

In [27]:
question = 'Can I still join the course?'
context = 'EMPTY'

In [28]:
prompt = prompt_template.format(question=question, context=context)

In [29]:
answer_json = llm(prompt)

In [31]:
answer = json.loads(answer_json)

In [32]:
answer

{'action': 'SEARCH',
 'reasoning': "The question about joining the course is likely addressed in the course's FAQ or enrollment information. Since the context is empty, I need to refer to the FAQ database to provide an accurate answer."}

In [33]:
def build_context(search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    return context.strip()

In [34]:
search_results = search(question)
context = build_context(search_results)
prompt = prompt_template.format(question=question, context=context)

In [35]:
answer_json = llm(prompt)

In [36]:
print(answer_json)

{
"action": "ANSWER",
"answer": "Yes, you can still join the course even after the start date. You are eligible to submit homeworks without registering, but be aware of the deadlines for final projects. It's advisable not to leave everything for the last minute.",
"source": "CONTEXT"
}


### Building Agentic Search

In [37]:
def dedup(seq):
    seen = set()
    result = []
    for el in seq:
        _id = el['_id']
        if _id in seen:
            continue
        seen.add(_id)
        result.append(el)
    return result

In [70]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than {max_iterations} iterations for a given student question.
The current iteration number: {iteration_number}. If we exceed the allowed number 
of iterations, give the best possible answer with the provided information.

Output templates return your answer strictly in this JSON format without extra text:

If you want to perform search, use this template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>",
"keywords": ["search query 1", "search query 2", ...]
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER_CONTEXT",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

<QUESTION>
{question}
</QUESTION>

<SEARCH_QUERIES>
{search_queries}
</SEARCH_QUERIES>

<CONTEXT> 
{context}
</CONTEXT>

<PREVIOUS_ACTIONS>
{previous_actions}
</PREVIOUS_ACTIONS>
""".strip()

In [71]:
question = 'how do I do well on module 1'
max_iterations = 3
iteration_number = 0
search_queries = []
search_results  = []
previous_actions = []

In [72]:
context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=max_iterations,
    iteration_number=iteration_number
)

In [73]:
answer_json = llm(prompt)

In [74]:
answer = json.loads(answer_json)

In [75]:
previous_actions.append(answer)

In [76]:
keywords = answer['keywords']

In [77]:
for kw in keywords:
    search_queries.append(kw)
    sr = search(kw)
    search_results.extend(sr)

In [78]:
search_results = dedup(search_results)

In [79]:
iteration_number = 2

context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=max_iterations,
    iteration_number=iteration_number
)

In [80]:
answer_json = llm(prompt)
answer = json.loads(answer_json)

In [81]:
print(answer['answer'])

To do well in Module 1, focus on understanding the foundational concepts thoroughly. Here are some general tips: 1) Review all course materials and lecture notes carefully. 2) Practice any hands-on exercises or labs multiple times to build confidence. 3) Don't hesitate to ask questions if concepts are unclear. 4) Manage your time effectively to avoid last-minute cramming. 5) Form study groups with classmates to discuss challenging topics. Since the context doesn't contain specific Module 1 success strategies, these general study approaches should help you perform well.


In [82]:
question = "what do I need to do to be successful at module 1?"

search_queries = []
search_results = []
previous_actions = []

iteration = 0

while True:
    print(f'ITERATION #{iteration}...')

    context = build_context(search_results)
    prompt = prompt_template.format(
        question=question,
        context=context,
        search_queries="\n".join(search_queries),
        previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
        max_iterations=3,
        iteration_number=iteration
    )

    print(prompt)

    answer_json = llm(prompt).strip()
    answer = json.loads(answer_json)
    print(json.dumps(answer, indent=2))

    previous_actions.append(answer)

    action = answer['action']
    if action != 'SEARCH':
        break

    keywords = answer['keywords']
    search_queries = list(set(search_queries) | set(keywords))
    
    for k in keywords:
        res = search(k)
        search_results.extend(res)

    search_results = dedup(search_results)
    
    iteration = iteration + 1
    if iteration >= 4:
        break

    print()

ITERATION #0...
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current 

In [83]:
answer

{'action': 'ANSWER',
 'answer': "To be successful in Module 1, which focuses on Docker and Terraform, you should ensure you have a good understanding of the basics of Docker and Terraform. Install Docker and Terraform on your machine, familiarize yourself with Docker commands and Terraform configurations, and practice creating and managing containers and infrastructure as code. Additionally, make sure to follow all course instructions carefully, complete all assignments, and participate in any discussions or forums to clarify doubts. If you encounter errors like 'ModuleNotFoundError', ensure all required Python modules like 'psycopg2' are installed correctly.",
 'source': 'OWN_KNOWLEDGE'}

In [85]:
iteration

1

### Function Calling and Tool Use

In [104]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [133]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [134]:
def do_call(tool_call):
    function_name = tool_call.function.name 
    arguments = json.loads(tool_call.function.arguments)

    f = globals()[function_name]
    result = f(**arguments)

    return {
        "tool_call_id": tool_call.id,
        "role": "tool",
        "name": function_name,
        "content": json.dumps(result, indent=2),
    }

In [ ]:
question = "How do I do well in module 1?"

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
If you look up something in FAQ, convert the student question into multiple queries.
""".strip()

tools = [search_tool]

chat_messages = [
    {"role": "system", "content": developer_prompt},
    {"role": "user", "content": question}
]


response = client.chat.completions.create(
    model="deepseek-chat",
    messages=chat_messages,
    tools=tools,
    tool_choice="auto"
)


In [ ]:
tool_calls = response.choices[0].message.tool_calls

if tool_calls:
    for call in tool_calls:
        result = do_call(call)
        chat_messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": call.id,
                "type": "function",
                "function": {
                    "name": call.function.name,
                    "arguments": call.function.arguments
                }
            }]
        })
        chat_messages.append(result)

    # Get the final response after tool calls
    final_response = client.chat.completions.create(
        model="deepseek-chat",
        messages=chat_messages
    )
    print(final_response.choices[0].message.content)
else:
    print(response.choices[0].message.content)

Based on the search results, here are some key tips to do well in Module 1 (Docker and Terraform) of the Data Engineering Zoomcamp:

1. **Docker Best Practices**:
   - Store your code in the default Linux distro (if using WSL2 on Windows) for better filesystem performance
   - Refer to Docker's official documentation for best practices

2. **Common Technical Issues to Be Aware Of**:
   - When working with Postgres/SQLAlchemy:
     - Use the correct connection string format: `postgresql+psycopg://` instead of just `postgresql://`
     - Make sure to install required Python packages (`psycopg2` or `psycopg2-binary`)
     - If you get module errors, try:
       ```bash
       pip install psycopg2-binary
       # or if already installed
       pip install psycopg2-binary --upgrade
       ```

3. **General Advice**:
   - Pay close attention to the installation instructions
   - Test your setup early to catch any environment issues
   - When you encounter errors, check the exact error messag